In [0]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col, current_timestamp, lit, to_date
import re

In [0]:
# Configuración de rutas
storage_account = "adlssmartdatamanuel95"
ruta_origen   = f"abfss://raw@{storage_account}.dfs.core.windows.net/"
tabla_destino = "adbsmartdatamanuelestrada.bronze.base_tiendas_brz"

print("="*60)
print("CONFIGURACIÓN DEL PIPELINE - TIENDAS")
print("="*60)
print(f"Ruta origen: {ruta_origen}")
print(f"Tabla destino: {tabla_destino}")
print("="*60)

In [0]:
print("="*60)
print("LECTURA DEL ARCHIVO CSV")
print("="*60)

# ============================================================
# PASO 1: Extraer mes del nombre del archivo
# ============================================================
archivos = dbutils.fs.ls(ruta_origen)
# Filtrar SOLO archivos que empiezan con "base_tiendas" y terminan en .csv
archivos_csv = [f.name for f in archivos if f.name.startswith('base_tiendas') and f.name.endswith('.csv')]

if len(archivos_csv) == 0:
    print(f"\n⚠️  No se encontró ningún archivo 'base_tiendas*.csv' en {ruta_origen}")
    print("ℹ️  Esto es normal - los archivos de tiendas no se actualizan en cada ciclo.")
    print("✅ Pipeline finalizado exitosamente (sin datos nuevos).\n")
    print("="*60)
    dbutils.jobs.taskValues.set(key="has_data", value="false")
    dbutils.notebook.exit("NO_DATA")

nombre_archivo = archivos_csv[0]
print(f"\nArchivo detectado: {nombre_archivo}")

# Extraer YYYYMM del nombre (ej: base_tiendas_202601.csv -> 202601)
match = re.search(r'(\d{6})', nombre_archivo)
if not match:
    raise ValueError(f"❌ ERROR: No se pudo extraer el mes del nombre del archivo: {nombre_archivo}")

mes_archivo = match.group(1)
ano_archivo = mes_archivo[:4]
mes_num_archivo = mes_archivo[4:6]
print(f"Mes del archivo: {mes_archivo} ({mes_num_archivo}/{ano_archivo})")

# Crear fecha_registro (primer día del mes)
fecha_registro = f"{ano_archivo}-{mes_num_archivo}-01"
print(f"Fecha registro: {fecha_registro}")

# ============================================================
# VALIDACIÓN #1: Verificar si el mes ya existe en la tabla
# ============================================================
print("\n" + "-"*60)
print("VALIDACIÓN #1: Control de Duplicados por Mes")
print("-"*60)

try:
    # Query directo sin materializar toda la tabla (optimizado para Serverless)
    count_mes = spark.sql(f"""
        SELECT COUNT(*) as total 
        FROM {tabla_destino} 
        WHERE fecha_registro = '{fecha_registro}'
    """).collect()[0]['total']
    
    if count_mes > 0:
        raise ValueError(
            f"❌ ERROR: El mes {mes_archivo} (fecha {fecha_registro}) ya existe en la tabla Bronze.\n"
            f"   No se puede cargar el mismo mes dos veces.\n"
            f"   Para recargar, primero elimine los datos del mes: \n"
            f"   DELETE FROM {tabla_destino} WHERE fecha_registro = '{fecha_registro}'"
        )
    
    print(f"✅ El mes {mes_archivo} NO existe en la tabla. Se procederá con la carga.")
    
except Exception as e:
    if "Table or view not found" in str(e) or "cannot be found" in str(e).lower():
        print(f"ℹ️  La tabla no existe. Se creará con los datos del mes {mes_archivo}.")
    else:
        raise

# ============================================================
# PASO 2: Leer el CSV
# ============================================================
print("\n" + "-"*60)
print("Lectura del archivo CSV")
print("-"*60)

# Construir ruta completa al archivo específico
ruta_completa = ruta_origen + nombre_archivo
print(f"Leyendo: {ruta_completa}")

df = (spark.read
  .format("csv")
  .option("header", "true")
  .option("delimiter", ";")
  .option("inferSchema", "false")
  .load(ruta_completa)
)

print(f"✅ CSV cargado: {len(df.columns)} columnas")
print(f"Columnas: {', '.join(df.columns)}")

# ============================================================
# VALIDACIÓN #2: Verificar que el DataFrame NO esté vacío
# ============================================================
print("\n" + "-"*60)
print("VALIDACIÓN #2: DataFrame Vacío")
print("-"*60)

total_registros = df.count()
print(f"Registros en el archivo: {total_registros:,}")

if total_registros == 0:
    raise ValueError(
        f"❌ ERROR: El archivo {nombre_archivo} está vacío.\n"
        f"   No se puede cargar un archivo sin registros."
    )

print(f"✅ El archivo contiene {total_registros:,} registros.")
print("\n" + "="*60)
print("✅ TODAS LAS VALIDACIONES APROBADAS")
print("="*60)

In [0]:
print("="*60)
print("ESTANDARIZACIÓN DE COLUMNAS")
print("="*60)

# Primero, obtener los nombres de columnas exactos tal como Spark los lee
columnas_originales = df.columns
print(f"\nColumnas originales leídas por Spark:")
for i, col_name in enumerate(columnas_originales, 1):
    print(f"  {i}. '{col_name}'")

# Agregar fecha_registro y timestamp de ingestión
df_with_dates = df \
  .withColumn("fecha_registro", lit(fecha_registro)) \
  .withColumn("data_ingestion_ts", current_timestamp().cast("string"))

# Renombrar columnas usando los nombres exactos
df_renamed = df_with_dates \
  .withColumnRenamed(columnas_originales[0], "cod_tienda") \
  .withColumnRenamed(columnas_originales[1], "pais") \
  .withColumnRenamed(columnas_originales[2], "ciudad")

# Reordenar columnas para poner fecha_registro al inicio
df_final = df_renamed.select(
    "fecha_registro",
    "cod_tienda",
    "pais",
    "ciudad",
    "data_ingestion_ts"
)

print(f"\nColumnas estandarizadas:")
for i, col_name in enumerate(df_final.columns, 1):
    print(f"  {i}. {col_name}")

print("\n✅ Estandarización completa")
print("="*60)

In [0]:
print("="*60)
print("ESCRITURA A TABLA BRONZE")
print("="*60)

print(f"\nRegistros a insertar: {total_registros:,}")
print(f"Mes: {mes_archivo}")
print(f"Fecha registro: {fecha_registro}")

df_final.write \
  .format("delta") \
  .mode("append") \
  .saveAsTable(tabla_destino)

print("\n" + "="*60)
print("✅ INSERCIÓN EXITOSA")
print("="*60)
print(f"Tabla: {tabla_destino}")
print(f"Mes insertado: {mes_archivo}")
print(f"Fecha registro: {fecha_registro}")
print(f"Registros insertados: {total_registros:,}")
# ⚡ Comentado para performance (full table scan costoso)
# print(f"Total registros en tabla: {spark.table(tabla_destino).count():,}")
print("="*60)

In [0]:
# Salir con código de éxito indicando que SÍ había datos
dbutils.jobs.taskValues.set(key="has_data", value="true")
dbutils.notebook.exit("HAS_DATA")